## What the Silver layer should accomplish

1. **Apply appropriate data types:** Convert numeric and timestamp columns from strings into their correct Spark data types, while keeping identifiers and ZIP code prefixes as strings.
2. **Normalize source values:** Trim whitespace and convert empty strings into NULL.
3. **Preserve business meaning:** Retain original business attributes and their meaning without inventing or silently correcting source values.
4. **Add data-quality flags:** Identify missing attributes and relevant data anomalies without unnecessarily rejecting records.
5. **Validate data conversions:** Detect invalid numeric and timestamp conversions before saving Silver tables.
6. **Validate data integrity:** Check primary keys, composite keys, duplicates where applicable, and other table-specific quality rules.
7. **Handle duplicates appropriately:** Remove exact duplicate geographic records where justified, but preserve legitimate repeated business relationships.
8. **Preserve problematic records:** Retain records with missing or anomalous optional attributes, using NULL values and quality flags rather than silently dropping them.
9. **Maintain ingestion metadata:** Retain ingestion_timestamp, source_file, and batch_id for traceability.
10. **Enforce consistent schemas:** Align every Silver table with its explicitly defined StructType schema and validate it before writing.

## TODO – Future Improvements

1. **Declarative Data Quality Expectations**
   - Explore Databricks Lakeflow Spark Declarative Pipelines expectations.
   - Implement appropriate expectations alongside existing data-quality flags.
   - Evaluate Great Expectations as an alternative or complementary testing framework.
   - Preserve the current data-retention strategy when choosing expectation actions.

2. **Incremental Data Processing**
   - Document the current full-refresh strategy (overwrite).
   - Implement append-based Bronze ingestion with batch tracking and idempotency.
   - Introduce incremental Silver transformations using Delta MERGE where appropriate.
   - Investigate late-arriving records, source updates, and duplicate ingestion.
   - Compare full-refresh and incremental processing approaches.

3. **Pipeline Orchestration**
   - Implement scheduled execution using Lakeflow Jobs.
   - Add dependencies between pipeline stages.
   - Implement monitoring and failure handling.

## Processing Strategy

The current Silver transformations use a full-refresh approach.

- Each Silver table is rebuilt from its corresponding Bronze Delta table.
- Data is read from Bronze, transformed, validated, and written using `mode("overwrite")`.
- The current implementation is designed for the static Olist historical dataset.
- Ingestion metadata (`ingestion_timestamp`, `source_file`, and `batch_id`) is retained for data lineage and traceability.

### Future Improvements

For future incremental processing:
- Introduce append-based Bronze ingestion with batch tracking and idempotency.
- Process newly ingested or changed records in Silver.
- Evaluate Delta MERGE operations for appropriate tables.
- Define handling for late-arriving records and source updates.

Incremental processing is intentionally outside the scope of the current implementation.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    BooleanType
)

# ORDERS

In [0]:
# Step 1. Define schema

orders_silver_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("source_file", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("flag_approval_before_purchase", BooleanType(), True),
    StructField("flag_carrier_before_approval", BooleanType(), True),
    StructField("flag_customer_before_carrier", BooleanType(), True)
])

In [0]:
# Step 2. Load Bronze and transform

from pyspark.sql import functions as F

orders_silver_df = spark.table("lakehouse.bronze.orders")

string_columns = [
    "order_id",
    "customer_id",
    "order_status"
]

timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Trim strings and replace empty strings with NULL
orders_silver_df = orders_silver_df.withColumns({
    column: F.when(
        F.trim(F.col(column)) == "",
        None
    ).otherwise(F.trim(F.col(column)))
    for column in string_columns + timestamp_columns
})

# Convert timestamp columns
orders_silver_df = orders_silver_df.withColumns({
    f"{column}_converted": F.to_timestamp(
        F.col(column),
        "yyyy-MM-dd HH:mm:ss"
    )
    for column in timestamp_columns
})

# Add timestamp sequence anomaly flags
orders_silver_df = orders_silver_df.withColumns({
    "flag_approval_before_purchase": F.when(
        F.col("order_approved_at_converted").isNull() |
        F.col("order_purchase_timestamp_converted").isNull(),
        None
    ).otherwise(
        F.col("order_approved_at_converted") <
        F.col("order_purchase_timestamp_converted")
    ),

    "flag_carrier_before_approval": F.when(
        F.col("order_delivered_carrier_date_converted").isNull() |
        F.col("order_approved_at_converted").isNull(),
        None
    ).otherwise(
        F.col("order_delivered_carrier_date_converted") <
        F.col("order_approved_at_converted")
    ),

    "flag_customer_before_carrier": F.when(
        F.col("order_delivered_customer_date_converted").isNull() |
        F.col("order_delivered_carrier_date_converted").isNull(),
        None
    ).otherwise(
        F.col("order_delivered_customer_date_converted") <
        F.col("order_delivered_carrier_date_converted")
    )
})

In [0]:
# Step 3. Validate conversions

conversion_columns = timestamp_columns

invalid_conversion_counts = orders_silver_df.agg(
    *[
        F.sum(
            F.when(
                F.col(column).isNotNull()
                & (F.trim(F.col(column)) != "")
                & F.col(f"{column}_converted").isNull(),
                1
            ).otherwise(0)
        ).alias(column)
        for column in conversion_columns
    ]
)

invalid_conversion_counts.display()

invalid_counts = invalid_conversion_counts.first().asDict()

total_invalid_conversions = sum(
    value or 0 for value in invalid_counts.values()
)

if total_invalid_conversions > 0:
    raise ValueError(
        f"Found {total_invalid_conversions} invalid conversions "
        "in Bronze orders. Silver table was not saved."
    )

order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,0,0,0,0


In [0]:
# Step 4. Replace converted columns

for column in timestamp_columns:
    orders_silver_df = (
        orders_silver_df
        .drop(column)
        .withColumn(column, F.col(f"{column}_converted"))
        .drop(f"{column}_converted")
    )

# Remove corrupt record column after validation
orders_silver_df = orders_silver_df.drop("_corrupt_record")

In [0]:
# Step 5. Validate data quality

quality_metrics = orders_silver_df.agg(
    F.count("*").alias("total_rows"),

    F.countDistinct("order_id").alias("distinct_order_ids"),

    F.sum(
        F.when(F.col("order_id").isNull(), 1).otherwise(0)
    ).alias("missing_order_ids"),

    F.sum(
        F.when(F.col("customer_id").isNull(), 1).otherwise(0)
    ).alias("missing_customer_ids"),

    F.sum(
        F.when(F.col("order_status").isNull(), 1).otherwise(0)
    ).alias("missing_order_status"),

    F.sum(
        F.when(F.col("order_purchase_timestamp").isNull(), 1).otherwise(0)
    ).alias("missing_purchase_timestamps"),

    F.sum(
        F.when(F.col("order_approved_at").isNull(), 1).otherwise(0)
    ).alias("missing_approval_timestamps"),

    F.sum(
        F.when(F.col("order_delivered_carrier_date").isNull(), 1).otherwise(0)
    ).alias("missing_carrier_timestamps"),

    F.sum(
        F.when(F.col("order_delivered_customer_date").isNull(), 1).otherwise(0)
    ).alias("missing_customer_delivery_timestamps"),

    F.sum(
        F.when(F.col("order_estimated_delivery_date").isNull(), 1).otherwise(0)
    ).alias("missing_estimated_delivery_timestamps"),

    F.sum(
        F.when(F.col("flag_approval_before_purchase") == True, 1).otherwise(0)
    ).alias("approval_before_purchase"),

    F.sum(
        F.when(F.col("flag_carrier_before_approval") == True, 1).otherwise(0)
    ).alias("carrier_before_approval"),

    F.sum(
        F.when(F.col("flag_customer_before_carrier") == True, 1).otherwise(0)
    ).alias("customer_before_carrier")
)

quality_metrics.display()

quality_counts = quality_metrics.first().asDict()

# Check duplicate order IDs
duplicate_order_ids = (
    orders_silver_df
    .groupBy("order_id")
    .count()
    .filter(
        F.col("order_id").isNotNull() &
        (F.col("count") > 1)
    )
    .count()
)

if quality_counts["missing_order_ids"] > 0:
    raise ValueError("Missing order_id values detected.")

if duplicate_order_ids > 0:
    raise ValueError("Duplicate order_id values detected.")

total_rows,distinct_order_ids,missing_order_ids,missing_customer_ids,missing_order_status,missing_purchase_timestamps,missing_approval_timestamps,missing_carrier_timestamps,missing_customer_delivery_timestamps,missing_estimated_delivery_timestamps,approval_before_purchase,carrier_before_approval,customer_before_carrier
99441,99441,0,0,0,0,160,1783,2965,0,0,1359,23


In [0]:
# Step 6. Align and validate schema

orders_silver_df = orders_silver_df.select(
    *orders_silver_schema.fieldNames()
)

if orders_silver_df.schema != orders_silver_schema:
    raise ValueError(
        "Silver DataFrame schema does not match the target schema."
    )

In [0]:
# Step 7. Save table

(
    orders_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lakehouse.silver.orders")
)

# CUSTOMERS

In [0]:
# Step 1. Define schema

customers_silver_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("source_file", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("flag_missing_customer_unique_id", BooleanType(), False),
    StructField("flag_missing_customer_zip_code_prefix", BooleanType(), False)
])

In [0]:
# Step 2. Load Bronze and transform

from pyspark.sql import functions as F

customers_silver_df = spark.table("lakehouse.bronze.customers")

string_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

# Trim strings and replace empty strings with NULL
customers_silver_df = customers_silver_df.withColumns({
    column: F.when(
        F.trim(F.col(column)) == "",
        None
    ).otherwise(F.trim(F.col(column)))
    for column in string_columns
})

# Normalize state abbreviations
customers_silver_df = customers_silver_df.withColumn(
    "customer_state",
    F.upper(F.col("customer_state"))
)

# Add missing-value flags
customers_silver_df = customers_silver_df.withColumns({
    "flag_missing_customer_unique_id":
        F.col("customer_unique_id").isNull(),

    "flag_missing_customer_zip_code_prefix":
        F.col("customer_zip_code_prefix").isNull()
})

In [0]:
# Step 3. Validate conversions
# Not applicable. No numeric or timestamp conversions are required for this table.

In [0]:
# Step 4. Replace converted columns
# Not applicable. No converted columns were created.

In [0]:
# Step 5. Validate data quality

customer_unique_id_groups = (
    customers_silver_df
    .filter(F.col("customer_unique_id").isNotNull())
    .groupBy("customer_unique_id")
    .agg(
        F.count("customer_id").alias("customer_id_count")
    )
)

customer_unique_metrics = customer_unique_id_groups.agg(
    F.sum(
        F.when(
            F.col("customer_id_count") > 1,
            1
        ).otherwise(0)
    ).alias("unique_customers_with_multiple_ids"),

    F.sum(
        F.when(
            F.col("customer_id_count") > 1,
            F.col("customer_id_count") - 1
        ).otherwise(0)
    ).alias("additional_customer_records"),

    F.max("customer_id_count").alias("max_customer_ids_per_unique_customer")
)

quality_metrics = customers_silver_df.agg(
    F.count("*").alias("total_rows"),

    F.countDistinct("customer_id").alias("distinct_customer_ids"),

    F.countDistinct("customer_unique_id").alias("distinct_customer_unique_ids"),

    F.sum(
        F.when(F.col("customer_id").isNull(), 1).otherwise(0)
    ).alias("missing_customer_ids"),

    F.sum(
        F.when(F.col("customer_unique_id").isNull(), 1).otherwise(0)
    ).alias("missing_customer_unique_ids"),

    F.sum(
        F.when(F.col("customer_zip_code_prefix").isNull(), 1).otherwise(0)
    ).alias("missing_customer_zip_code_prefix")
).crossJoin(customer_unique_metrics)

quality_metrics.display()

quality_counts = quality_metrics.first().asDict()

# Check duplicate primary keys
duplicate_customer_ids = (
    customers_silver_df
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("customer_id").isNotNull() &
        (F.col("count") > 1)
    )
    .count()
)

# Check missing-value flag consistency
flag_validation = customers_silver_df.filter(
    (
        F.col("flag_missing_customer_unique_id") !=
        F.col("customer_unique_id").isNull()
    ) |
    (
        F.col("flag_missing_customer_zip_code_prefix") !=
        F.col("customer_zip_code_prefix").isNull()
    )
).count()

if quality_counts["missing_customer_ids"] > 0:
    raise ValueError("Missing customer_id values detected.")

if duplicate_customer_ids > 0:
    raise ValueError("Duplicate customer_id values detected.")

if flag_validation > 0:
    raise ValueError("Missing-value flag inconsistencies detected.")

total_rows,distinct_customer_ids,distinct_customer_unique_ids,missing_customer_ids,missing_customer_unique_ids,missing_customer_zip_code_prefix,unique_customers_with_multiple_ids,additional_customer_records,max_customer_ids_per_unique_customer
99441,99441,96096,0,0,0,2997,3345,17


In [0]:
# Step 6. Align and validate schema

customers_silver_df = customers_silver_df.select(
    *customers_silver_schema.fieldNames()
)

if customers_silver_df.schema != customers_silver_schema:
    raise ValueError(
        "Silver DataFrame schema does not match the target schema."
    )

In [0]:
# Step 7. Save table

(
    customers_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lakehouse.silver.customers")
)

# NOTES:
# 96,096 distinct underlying customers.
# 2,997 underlying customers have multiple customer_id values.
# 3,345 additional customer records beyond the first record per unique customer.
# The largest group has 17 customer IDs.

# PRODUCTS

In [0]:
# Step 1. Define the target schema

products_silver_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", DoubleType(), True),
    StructField("product_length_cm", DoubleType(), True),
    StructField("product_height_cm", DoubleType(), True),
    StructField("product_width_cm", DoubleType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("source_file", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("flag_missing_product_category_name", BooleanType(), False),
    StructField("flag_missing_descriptive_attributes", BooleanType(), False),
    StructField("flag_missing_physical_attributes", BooleanType(), False)
])

In [0]:
# Step 2. Load Bronze and transform

from pyspark.sql import functions as F

products_silver_df = spark.table("lakehouse.bronze.products")

string_columns = [
    "product_id",
    "product_category_name"
]

integer_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

double_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

numeric_columns = integer_columns + double_columns

# Normalize strings and numeric source values
products_silver_df = products_silver_df.withColumns({
    column: F.when(
        F.trim(F.col(column)) == "",
        None
    ).otherwise(F.trim(F.col(column)))
    for column in string_columns + numeric_columns
})

# Create converted integer columns
products_silver_df = products_silver_df.withColumns({
    f"{column}_converted": F.expr(f"try_cast({column} AS INT)")
    for column in integer_columns
})

# Create converted double columns
products_silver_df = products_silver_df.withColumns({
    f"{column}_converted": F.expr(f"try_cast({column} AS DOUBLE)")
    for column in double_columns
})

# Add missing-value flags
products_silver_df = products_silver_df.withColumns({
    "flag_missing_product_category_name":
        F.col("product_category_name").isNull(),

    "flag_missing_descriptive_attributes":
        (
            F.col("product_name_lenght_converted").isNull() &
            F.col("product_description_lenght_converted").isNull() &
            F.col("product_photos_qty_converted").isNull()
        ),

    "flag_missing_physical_attributes":
        (
            F.col("product_weight_g_converted").isNull() &
            F.col("product_length_cm_converted").isNull() &
            F.col("product_height_cm_converted").isNull() &
            F.col("product_width_cm_converted").isNull()
        )
})

In [0]:
# Step 3. Validate conversions

conversion_columns = numeric_columns

invalid_conversion_counts = products_silver_df.agg(
    *[
        F.sum(
            F.when(
                F.col(column).isNotNull()
                & (F.trim(F.col(column)) != "")
                & F.col(f"{column}_converted").isNull(),
                1
            ).otherwise(0)
        ).alias(column)
        for column in conversion_columns
    ]
)

invalid_conversion_counts.display()

invalid_counts = invalid_conversion_counts.first().asDict()

total_invalid_conversions = sum(
    value or 0 for value in invalid_counts.values()
)

if total_invalid_conversions > 0:
    raise ValueError(
        f"Found {total_invalid_conversions} invalid conversions "
        "in Bronze products. Silver table was not saved."
    )

product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,0,0,0,0,0,0


In [0]:
# Step 4. Replace converted columns

for column in numeric_columns:
    products_silver_df = (
        products_silver_df
        .drop(column)
        .withColumn(column, F.col(f"{column}_converted"))
        .drop(f"{column}_converted")
    )

# Remove corrupt record column after validation
products_silver_df = products_silver_df.drop("_corrupt_record")

In [0]:
# Step 5. Validate data quality

quality_metrics = products_silver_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("product_id").alias("distinct_product_ids"),
    F.sum(
        F.when(F.col("product_id").isNull(), 1).otherwise(0)
    ).alias("missing_product_ids"),
    F.sum(
        F.when(F.col("product_category_name").isNull(), 1).otherwise(0)
    ).alias("missing_product_categories"),
    F.sum(
        F.when(F.col("flag_missing_descriptive_attributes"), 1).otherwise(0)
    ).alias("missing_descriptive_attributes"),
    F.sum(
        F.when(F.col("flag_missing_physical_attributes"), 1).otherwise(0)
    ).alias("missing_physical_attributes")
)

quality_metrics.display()

quality_counts = quality_metrics.first().asDict()

duplicate_product_ids = (
    products_silver_df
    .groupBy("product_id")
    .count()
    .filter(
        F.col("product_id").isNotNull() &
        (F.col("count") > 1)
    )
    .count()
)

if quality_counts["missing_product_ids"] > 0:
    raise ValueError("Missing product_id values detected.")

if duplicate_product_ids > 0:
    raise ValueError("Duplicate product_id values detected.")

total_rows,distinct_product_ids,missing_product_ids,missing_product_categories,missing_descriptive_attributes,missing_physical_attributes
32951,32951,0,610,610,2


In [0]:
# Step 6. Align and validate schema

products_silver_df = products_silver_df.select(
    *products_silver_schema.fieldNames()
)

if products_silver_df.schema != products_silver_schema:
    raise ValueError(
        "Silver DataFrame schema does not match the target schema."
    )

In [0]:
# Step 7. Save table

(
    products_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lakehouse.silver.products")
)

# ORDER_ITEMS

In [0]:
order_items_silver_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("source_file", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("flag_missing_product_id", BooleanType(), False),
    StructField("flag_missing_seller_id", BooleanType(), False),
    StructField("flag_missing_shipping_limit_date", BooleanType(), False),
    StructField("flag_missing_price", BooleanType(), False),
    StructField("flag_missing_freight_value", BooleanType(), False)
])

In [0]:
order_items_silver_df = spark.table("lakehouse.bronze.order_items")

# Define column groups
string_columns = [
    "order_id",
    "product_id",
    "seller_id"
]

integer_columns = [
    "order_item_id"
]

double_columns = [
    "price",
    "freight_value"
]

timestamp_columns = [
    "shipping_limit_date"
]

numeric_columns = integer_columns + double_columns

# Normalize strings and convert source columns
order_items_silver_df = (
    order_items_silver_df

    # Trim whitespace and convert empty strings to NULL
    .withColumns({
        column: F.when(
            F.trim(F.col(column)) == "",
            None
        ).otherwise(F.trim(F.col(column)))
        for column in string_columns + numeric_columns + timestamp_columns
    })

    # Convert integer columns
    .withColumns({
        f"{column}_converted": F.expr(
            f"try_cast(`{column}` AS INT)"
        )
        for column in integer_columns
    })

    # Convert double columns
    .withColumns({
        f"{column}_converted": F.expr(
            f"try_cast(`{column}` AS DOUBLE)"
        )
        for column in double_columns
    })

    # Convert timestamp columns
    .withColumns({
        f"{column}_converted": F.expr(
            f"try_cast(`{column}` AS TIMESTAMP)"
        )
        for column in timestamp_columns
    })

    # Add missing-value flags
    .withColumns({
        "flag_missing_product_id": (
            F.col("product_id").isNull()
        ),

        "flag_missing_seller_id": (
            F.col("seller_id").isNull()
        ),

        "flag_missing_shipping_limit_date": (
            F.col("shipping_limit_date_converted").isNull()
        ),

        "flag_missing_price": (
            F.col("price_converted").isNull()
        ),

        "flag_missing_freight_value": (
            F.col("freight_value_converted").isNull()
        )
    })
)

In [0]:
conversion_columns = numeric_columns + timestamp_columns

invalid_conversion_counts = order_items_silver_df.agg(
    *[
        F.sum(
            F.when(
                F.col(column).isNotNull()
                & (F.trim(F.col(column)) != "")
                & F.col(f"{column}_converted").isNull(),
                1
            ).otherwise(0)
        ).alias(column)
        for column in conversion_columns
    ]
)

invalid_conversion_counts.display()

invalid_counts = invalid_conversion_counts.first().asDict()

total_invalid_conversions = sum(
    value or 0 for value in invalid_counts.values()
)

if total_invalid_conversions > 0:
    raise ValueError(
        f"Found {total_invalid_conversions} invalid conversions "
        "in Bronze order_items. Silver table was not saved."
    )

order_item_id,price,freight_value,shipping_limit_date
0,0,0,0


In [0]:
order_items_silver_df = (
    order_items_silver_df
    .drop(*conversion_columns)
)

for column in conversion_columns:
    order_items_silver_df = (
        order_items_silver_df
        .withColumn(column, F.col(f"{column}_converted"))
        .drop(f"{column}_converted")
    )

order_items_silver_df = order_items_silver_df.drop("_corrupt_record")

In [0]:
order_items_quality_metrics = order_items_silver_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(
            F.col("order_id").isNull()
            | F.col("order_item_id").isNull(),
            1
        ).otherwise(0)
    ).alias("missing_primary_key_values"),

    F.sum(
        F.when(F.col("flag_missing_product_id"), 1).otherwise(0)
    ).alias("missing_product_ids"),

    F.sum(
        F.when(F.col("flag_missing_seller_id"), 1).otherwise(0)
    ).alias("missing_seller_ids"),

    F.sum(
        F.when(F.col("flag_missing_shipping_limit_date"), 1).otherwise(0)
    ).alias("missing_shipping_limit_dates"),

    F.sum(
        F.when(F.col("flag_missing_price"), 1).otherwise(0)
    ).alias("missing_prices"),

    F.sum(
        F.when(F.col("flag_missing_freight_value"), 1).otherwise(0)
    ).alias("missing_freight_values")
)

order_items_quality_metrics.display()

quality_metrics = order_items_quality_metrics.first().asDict()

if quality_metrics["missing_primary_key_values"] > 0:
    raise ValueError(
        "Silver order_items contains missing primary key values."
    )

duplicate_keys = (
    order_items_silver_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_keys > 0:
    raise ValueError(
        f"Silver order_items contains {duplicate_keys} duplicate "
        "composite primary keys."
    )

total_rows,missing_primary_key_values,missing_product_ids,missing_seller_ids,missing_shipping_limit_dates,missing_prices,missing_freight_values
112650,0,0,0,0,0,0


In [0]:
order_items_silver_df = order_items_silver_df.select(
    *order_items_silver_schema.fieldNames()
)

if order_items_silver_df.schema != order_items_silver_schema:
    raise ValueError(
        "Silver order_items schema does not match the expected schema."
    )

In [0]:
(
    order_items_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lakehouse.silver.order_items")
)

# ORDER_PAYMENTS

In [0]:
order_payments_silver_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_sequential", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value", DoubleType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("source_file", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("flag_missing_payment_type", BooleanType(), False),
    StructField("flag_missing_payment_installments", BooleanType(), False),
    StructField("flag_missing_payment_value", BooleanType(), False)
])

In [0]:
order_payments_silver_df = spark.table("lakehouse.bronze.order_payments")

# Define column groups
string_columns = [
    "order_id",
    "payment_type"
]

integer_columns = [
    "payment_sequential",
    "payment_installments"
]

double_columns = [
    "payment_value"
]

numeric_columns = integer_columns + double_columns

# Normalize strings and convert source columns
order_payments_silver_df = (
    order_payments_silver_df

    # Trim whitespace and convert empty strings to NULL
    .withColumns({
        column: F.when(
            F.trim(F.col(column)) == "",
            None
        ).otherwise(F.trim(F.col(column)))
        for column in string_columns + numeric_columns
    })

    # Convert integer columns
    .withColumns({
        f"{column}_converted": F.expr(
            f"try_cast(`{column}` AS INT)"
        )
        for column in integer_columns
    })

    # Convert double columns
    .withColumns({
        f"{column}_converted": F.expr(
            f"try_cast(`{column}` AS DOUBLE)"
        )
        for column in double_columns
    })

    # Add missing-value flags
    .withColumns({
        "flag_missing_payment_type": (
            F.col("payment_type").isNull()
        ),

        "flag_missing_payment_installments": (
            F.col("payment_installments_converted").isNull()
        ),

        "flag_missing_payment_value": (
            F.col("payment_value_converted").isNull()
        )
    })
)

In [0]:
invalid_conversion_counts = order_payments_silver_df.agg(
    *[
        F.sum(
            F.when(
                F.col(column).isNotNull()
                & (F.trim(F.col(column)) != "")
                & F.col(f"{column}_converted").isNull(),
                1
            ).otherwise(0)
        ).alias(column)
        for column in numeric_columns
    ]
)

invalid_conversion_counts.display()

invalid_counts = invalid_conversion_counts.first().asDict()

total_invalid_conversions = sum(
    value or 0 for value in invalid_counts.values()
)

if total_invalid_conversions > 0:
    raise ValueError(
        f"Found {total_invalid_conversions} invalid numeric conversions "
        "in Bronze payments. Silver table was not saved."
    )

payment_sequential,payment_installments,payment_value
0,0,0


In [0]:
order_payments_silver_df = (
    order_payments_silver_df
    .drop(*numeric_columns)
)

for column in numeric_columns:
    order_payments_silver_df = (
        order_payments_silver_df
        .withColumn(column, F.col(f"{column}_converted"))
        .drop(f"{column}_converted")
    )

order_payments_silver_df = order_payments_silver_df.drop("_corrupt_record")

In [0]:
payments_quality_metrics = order_payments_silver_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(
            F.col("order_id").isNull()
            | F.col("payment_sequential").isNull(),
            1
        ).otherwise(0)
    ).alias("missing_primary_key_values"),

    F.sum(
        F.when(F.col("flag_missing_payment_type"), 1).otherwise(0)
    ).alias("missing_payment_types"),

    F.sum(
        F.when(F.col("flag_missing_payment_installments"), 1).otherwise(0)
    ).alias("missing_payment_installments"),

    F.sum(
        F.when(F.col("flag_missing_payment_value"), 1).otherwise(0)
    ).alias("missing_payment_values")
)

payments_quality_metrics.display()

quality_metrics = payments_quality_metrics.first().asDict()

if quality_metrics["missing_primary_key_values"] > 0:
    raise ValueError(
        "Silver payments contains missing primary key values."
    )

duplicate_keys = (
    order_payments_silver_df
    .groupBy("order_id", "payment_sequential")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_keys > 0:
    raise ValueError(
        f"Silver payments contains {duplicate_keys} duplicate "
        "composite primary keys."
    )

total_rows,missing_primary_key_values,missing_payment_types,missing_payment_installments,missing_payment_values
103886,0,0,0,0


In [0]:
payments_silver_df = order_payments_silver_df.select(
    *order_payments_silver_schema.fieldNames()
)

if payments_silver_df.schema != order_payments_silver_schema:
    raise ValueError(
        "Silver payments schema does not match the expected schema."
    )

In [0]:
(
    order_payments_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lakehouse.silver.order_payments")
)

# ORDER_REVIEWS

In [0]:
order_reviews_silver_schema = StructType([
    StructField("review_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("review_score", IntegerType(), True),
    StructField("review_comment_title", StringType(), True),
    StructField("review_comment_message", StringType(), True),
    StructField("review_creation_date", TimestampType(), True),
    StructField("review_answer_timestamp", TimestampType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("source_file", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("flag_missing_order_id", BooleanType(), False),
    StructField("flag_missing_review_score", BooleanType(), False),
    StructField("flag_missing_review_creation_date", BooleanType(), False),
    StructField("flag_missing_review_answer_timestamp", BooleanType(), False)
])

In [0]:
order_reviews_silver_df = spark.table("lakehouse.bronze.order_reviews")

# Define column groups
string_columns = [
    "review_id",
    "order_id",
    "review_comment_title",
    "review_comment_message"
]

integer_columns = [
    "review_score"
]

timestamp_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

numeric_columns = integer_columns

# Normalize strings and convert source columns
order_reviews_silver_df = (
    order_reviews_silver_df

    # Trim whitespace and convert empty strings to NULL
    .withColumns({
        column: F.when(
            F.trim(F.col(column)) == "",
            None
        ).otherwise(F.trim(F.col(column)))
        for column in string_columns + numeric_columns + timestamp_columns
    })

    # Convert integer columns
    .withColumns({
        f"{column}_converted": F.expr(
            f"try_cast(`{column}` AS INT)"
        )
        for column in integer_columns
    })

    # Convert timestamp columns
    .withColumns({
        f"{column}_converted": F.expr(
            f"try_cast(`{column}` AS TIMESTAMP)"
        )
        for column in timestamp_columns
    })

    # Add data-quality flags
    .withColumns({
        "flag_missing_order_id": (
            F.col("order_id").isNull()
        ),

        "flag_missing_review_score": (
            F.col("review_score_converted").isNull()
        ),

        "flag_missing_review_creation_date": (
            F.col("review_creation_date_converted").isNull()
        ),

        "flag_missing_review_answer_timestamp": (
            F.col("review_answer_timestamp_converted").isNull()
        )
    })
)

In [0]:
conversion_columns = numeric_columns + timestamp_columns

invalid_conversion_counts = order_reviews_silver_df.agg(
    *[
        F.sum(
            F.when(
                F.col(column).isNotNull()
                & (F.trim(F.col(column)) != "")
                & F.col(f"{column}_converted").isNull(),
                1
            ).otherwise(0)
        ).alias(column)
        for column in conversion_columns
    ]
)

invalid_conversion_counts.display()

invalid_counts = invalid_conversion_counts.first().asDict()

total_invalid_conversions = sum(
    value or 0 for value in invalid_counts.values()
)

if total_invalid_conversions > 0:
    raise ValueError(
        f"Found {total_invalid_conversions} invalid conversions "
        "in Bronze order_reviews. Silver table was not saved."
    )

review_score,review_creation_date,review_answer_timestamp
0,0,0


In [0]:
order_reviews_silver_df = (
    order_reviews_silver_df
    .drop(*conversion_columns)
)

for column in conversion_columns:
    order_reviews_silver_df = (
        order_reviews_silver_df
        .withColumn(column, F.col(f"{column}_converted"))
        .drop(f"{column}_converted")
    )

order_reviews_silver_df = order_reviews_silver_df.drop("_corrupt_record")

In [0]:
order_reviews_quality_metrics = order_reviews_silver_df.agg(
    F.count("*").alias("total_rows"),

    F.countDistinct(
        F.struct("order_id", "review_id")
    ).alias("distinct_composite_keys"),

    F.sum(
        F.when(F.col("review_id").isNull(), 1).otherwise(0)
    ).alias("missing_review_ids"),

    F.sum(
        F.when(F.col("order_id").isNull(), 1).otherwise(0)
    ).alias("missing_order_ids"),

    F.sum(
        F.when(F.col("flag_missing_review_score"), 1).otherwise(0)
    ).alias("missing_review_scores"),

    F.sum(
        F.when(F.col("flag_missing_review_creation_date"), 1).otherwise(0)
    ).alias("missing_review_creation_dates"),

    F.sum(
        F.when(F.col("flag_missing_review_answer_timestamp"), 1).otherwise(0)
    ).alias("missing_review_answer_timestamps")
)

order_reviews_quality_metrics.display()

quality_metrics = order_reviews_quality_metrics.first().asDict()

if quality_metrics["missing_review_ids"] > 0:
    raise ValueError("Silver order_reviews contains missing review IDs.")

if quality_metrics["missing_order_ids"] > 0:
    raise ValueError("Silver order_reviews contains missing order IDs.")

if quality_metrics["total_rows"] != quality_metrics["distinct_composite_keys"]:
    raise ValueError(
        "Silver order_reviews contains duplicate composite primary keys."
    )

total_rows,distinct_composite_keys,missing_review_ids,missing_order_ids,missing_review_scores,missing_review_creation_dates,missing_review_answer_timestamps
99224,99224,0,0,0,0,0


In [0]:
order_reviews_silver_df = order_reviews_silver_df.select(
    *order_reviews_silver_schema.fieldNames()
)

if order_reviews_silver_df.schema != order_reviews_silver_schema:
    raise ValueError(
        "Silver order_reviews schema does not match the expected schema."
    )

In [0]:
(
    order_reviews_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lakehouse.silver.order_reviews")
)

# NOTES:
# review_id is reused across multiple orders in the supplied Olist dataset, despite having consistent review attributes.

# SELLERS

In [0]:
sellers_silver_schema = StructType([
    StructField("seller_id", StringType(), True),
    StructField("seller_zip_code_prefix", StringType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("source_file", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("flag_missing_seller_zip_code_prefix", BooleanType(), False),
    StructField("flag_missing_seller_city", BooleanType(), False),
    StructField("flag_missing_seller_state", BooleanType(), False)
])

In [0]:
sellers_silver_df = spark.table("lakehouse.bronze.sellers")

string_columns = [
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
]

sellers_silver_df = (
    sellers_silver_df

    # Trim whitespace and convert empty strings to NULL
    .withColumns({
        column: F.when(
            F.trim(F.col(column)) == "",
            None
        ).otherwise(F.trim(F.col(column)))
        for column in string_columns
    })

    # Normalize state abbreviations
    .withColumn(
        "seller_state",
        F.upper(F.col("seller_state"))
    )

    # Create data-quality flags
    .withColumns({
        "flag_missing_seller_zip_code_prefix": (
            F.col("seller_zip_code_prefix").isNull()
        ),
        "flag_missing_seller_city": (
            F.col("seller_city").isNull()
        ),
        "flag_missing_seller_state": (
            F.col("seller_state").isNull()
        )
    })
)

In [0]:
# Step 3. Validate conversions
# Not applicable
# There are no numeric or timestamp business columns in the sellers source table that need conversion. All four business columns remain strings.
# The ingestion metadata (ingestion_timestamp, source_file, batch_id) is already retained from Bronze.

In [0]:
# Step 4. Replace converted columns
# Not applicable No temporary converted columns were created in Step 2, so there are no columns to replace.

In [0]:
sellers_quality_metrics = sellers_silver_df.agg(
    F.count("*").alias("total_rows"),

    F.countDistinct("seller_id").alias("distinct_seller_ids"),

    F.sum(
        F.when(F.col("seller_id").isNull(), 1).otherwise(0)
    ).alias("missing_seller_ids"),

    F.sum(
        F.when(
            F.col("flag_missing_seller_zip_code_prefix"), 1
        ).otherwise(0)
    ).alias("missing_zip_code_prefixes"),

    F.sum(
        F.when(
            F.col("flag_missing_seller_city"), 1
        ).otherwise(0)
    ).alias("missing_cities"),

    F.sum(
        F.when(
            F.col("flag_missing_seller_state"), 1
        ).otherwise(0)
    ).alias("missing_states")
)

sellers_quality_metrics.display()

quality_metrics = sellers_quality_metrics.first().asDict()

if quality_metrics["missing_seller_ids"] > 0:
    raise ValueError(
        "Silver sellers contains missing seller IDs."
    )

if quality_metrics["total_rows"] != quality_metrics["distinct_seller_ids"]:
    raise ValueError(
        "Silver sellers contains duplicate seller IDs."
    )

total_rows,distinct_seller_ids,missing_seller_ids,missing_zip_code_prefixes,missing_cities,missing_states
3095,3095,0,0,0,0


In [0]:
sellers_silver_df = sellers_silver_df.select(
    *sellers_silver_schema.fieldNames()
)

if sellers_silver_df.schema != sellers_silver_schema:
    raise ValueError(
        "Silver sellers schema does not match the expected schema."
    )

In [0]:
(
    sellers_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lakehouse.silver.sellers")
)

# GEOLOCATION

In [0]:
# Step 1. Define the target schema

geolocation_silver_schema = StructType([
    StructField("geolocation_zip_code_prefix", StringType(), True),
    StructField("geolocation_lat", DoubleType(), True),
    StructField("geolocation_lng", DoubleType(), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("source_file", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("flag_missing_zip_code_prefix", BooleanType(), False),
    StructField("flag_missing_latitude", BooleanType(), False),
    StructField("flag_missing_longitude", BooleanType(), False),
    StructField("flag_missing_city", BooleanType(), False),
    StructField("flag_missing_state", BooleanType(), False)
])

In [0]:
# Step 2. Load Bronze and transform

geolocation_silver_df = spark.table("lakehouse.bronze.geolocation")

string_columns = [
    "geolocation_zip_code_prefix",
    "geolocation_city",
    "geolocation_state"
]

numeric_columns = [
    "geolocation_lat",
    "geolocation_lng"
]

geolocation_silver_df = (
    geolocation_silver_df

    # Trim strings and convert empty strings to NULL
    .withColumns({
        column: F.when(
            F.trim(F.col(column)) == "",
            None
        ).otherwise(F.trim(F.col(column)))
        for column in string_columns + numeric_columns
    })

    # Convert coordinate columns to DOUBLE
    .withColumns({
        f"{column}_converted": F.expr(
            f"try_cast(`{column}` AS DOUBLE)"
        )
        for column in numeric_columns
    })

    # Normalize state abbreviations
    .withColumn(
        "geolocation_state",
        F.upper(F.col("geolocation_state"))
    )

    # Create data-quality flags
    .withColumns({
        "flag_missing_zip_code_prefix": (
            F.col("geolocation_zip_code_prefix").isNull()
        ),
        "flag_missing_latitude": (
            F.col("geolocation_lat_converted").isNull()
        ),
        "flag_missing_longitude": (
            F.col("geolocation_lng_converted").isNull()
        ),
        "flag_missing_city": (
            F.col("geolocation_city").isNull()
        ),
        "flag_missing_state": (
            F.col("geolocation_state").isNull()
        )
    })
)

In [0]:
# Step 3. Validate conversions

invalid_conversion_counts = geolocation_silver_df.agg(
    *[
        F.sum(
            F.when(
                F.col(column).isNotNull()
                & (F.trim(F.col(column)) != "")
                & F.col(f"{column}_converted").isNull(),
                1
            ).otherwise(0)
        ).alias(column)
        for column in numeric_columns
    ]
)

invalid_conversion_counts.display()

invalid_counts = invalid_conversion_counts.first().asDict()

total_invalid_conversions = sum(
    value or 0 for value in invalid_counts.values()
)

if total_invalid_conversions > 0:
    raise ValueError(
        f"Found {total_invalid_conversions} invalid coordinate conversions. "
        "Silver geolocation was not saved."
    )

geolocation_lat,geolocation_lng
0,0


In [0]:
# Step 4. Replace source columns with converted columns

geolocation_silver_df = geolocation_silver_df.drop(
    *numeric_columns
)

for column in numeric_columns:
    geolocation_silver_df = (
        geolocation_silver_df
        .withColumn(column, F.col(f"{column}_converted"))
        .drop(f"{column}_converted")
    )

geolocation_silver_df = geolocation_silver_df.drop("_corrupt_record")

In [0]:
# Step 5. Inspect initial data quality

geolocation_quality_metrics = geolocation_silver_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("flag_missing_zip_code_prefix"), 1).otherwise(0)
    ).alias("missing_zip_code_prefixes"),

    F.sum(
        F.when(F.col("flag_missing_latitude"), 1).otherwise(0)
    ).alias("missing_latitudes"),

    F.sum(
        F.when(F.col("flag_missing_longitude"), 1).otherwise(0)
    ).alias("missing_longitudes"),

    F.sum(
        F.when(F.col("flag_missing_city"), 1).otherwise(0)
    ).alias("missing_cities"),

    F.sum(
        F.when(F.col("flag_missing_state"), 1).otherwise(0)
    ).alias("missing_states"),

    F.min("geolocation_lat").alias("min_latitude"),
    F.max("geolocation_lat").alias("max_latitude"),
    F.min("geolocation_lng").alias("min_longitude"),
    F.max("geolocation_lng").alias("max_longitude")
)

geolocation_quality_metrics.display()

total_rows,missing_zip_code_prefixes,missing_latitudes,missing_longitudes,missing_cities,missing_states,min_latitude,max_latitude,min_longitude,max_longitude
1000163,0,0,0,0,0,-36.6053744107061,45.06593318269697,-101.46676644931476,121.10539381057764


In [0]:
# Step 6. Remove exact duplicate geographic records

geolocation_duplicate_metrics = (
    geolocation_silver_df
    .groupBy(
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    )
    .count()
    .agg(
        F.count("*").alias("distinct_geographic_records"),
        F.sum(
            F.when(F.col("count") > 1, 1).otherwise(0)
        ).alias("duplicate_groups"),
        F.sum(
            F.when(F.col("count") > 1, F.col("count") - 1).otherwise(0)
        ).alias("excess_duplicate_rows")
    )
)

geolocation_duplicate_metrics.display()

geolocation_silver_df = (
    geolocation_silver_df
    .dropDuplicates([
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ])
)

distinct_geographic_records,duplicate_groups,excess_duplicate_rows
738332,128174,261831


In [0]:
# Step 7. Validate duplicate removal

remaining_duplicates = (
    geolocation_silver_df
    .groupBy(
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if remaining_duplicates > 0:
    raise ValueError(
        "Silver geolocation still contains duplicate geographic records."
    )

geolocation_silver_df.agg(
    F.count("*").alias("rows_after_deduplication")
).display()

rows_after_deduplication
738332


In [0]:
# Step 8. Validate coordinate `ranges`

invalid_coordinate_ranges = geolocation_silver_df.filter(
    (F.col("geolocation_lat").isNotNull()) &
    (
        (F.col("geolocation_lat") < -34) |
        (F.col("geolocation_lat") > 6)
    )
    |
    (F.col("geolocation_lng").isNotNull()) &
    (
        (F.col("geolocation_lng") < -74) |
        (F.col("geolocation_lng") > -34)
    )
)

invalid_coordinate_ranges.count()

33

In [0]:
# Step 9. Align and validate schema

geolocation_silver_df = geolocation_silver_df.select(
    *geolocation_silver_schema.fieldNames()
)

if geolocation_silver_df.schema != geolocation_silver_schema:
    raise ValueError(
        "Silver geolocation schema does not match the expected schema."
    )

In [0]:
# Step 10. Save the Silver table

(
    geolocation_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lakehouse.silver.geolocation")
)

# product_category_name_translation

In [0]:
# Step 1. Define the target schema

product_category_translation_silver_schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("source_file", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("flag_missing_product_category_name", BooleanType(), False),
    StructField("flag_missing_product_category_name_english", BooleanType(), False)
])

In [0]:
# Step 2. Load Bronze and transform

product_category_translation_silver_df = spark.table(
    "lakehouse.bronze.product_category_name_translation"
)

string_columns = [
    "product_category_name",
    "product_category_name_english"
]

product_category_translation_silver_df = (
    product_category_translation_silver_df

    # Trim whitespace and convert empty strings to NULL
    .withColumns({
        column: F.when(
            F.trim(F.col(column)) == "",
            None
        ).otherwise(F.trim(F.col(column)))
        for column in string_columns
    })

    # Normalize category names
    .withColumns({
        column: F.lower(F.col(column))
        for column in string_columns
    })

    # Create data-quality flags
    .withColumns({
        "flag_missing_product_category_name": (
            F.col("product_category_name").isNull()
        ),
        "flag_missing_product_category_name_english": (
            F.col("product_category_name_english").isNull()
        )
    })
)

In [0]:
# Step 3. Validate conversions
# Not applicable
# Both business columns are strings, so no numeric or timestamp conversions are necessary.

In [0]:
# Step 4. Replace converted columns
# Not applicable
# No temporary converted columns were created.

In [0]:
product_category_translation_quality_metrics = (
    product_category_translation_silver_df.agg(
        F.count("*").alias("total_rows"),

        F.countDistinct(
            "product_category_name"
        ).alias("distinct_category_names"),

        F.sum(
            F.when(
                F.col("flag_missing_product_category_name"), 1
            ).otherwise(0)
        ).alias("missing_category_names"),

        F.sum(
            F.when(
                F.col("flag_missing_product_category_name_english"), 1
            ).otherwise(0)
        ).alias("missing_english_translations"),

        F.countDistinct(
            "product_category_name_english"
        ).alias("distinct_english_translations")
    )
)

product_category_translation_quality_metrics.display()

quality_metrics = (
    product_category_translation_quality_metrics.first().asDict()
)

if quality_metrics["missing_category_names"] > 0:
    raise ValueError(
        "Silver product category translation contains missing category names."
    )

if (
    quality_metrics["total_rows"]
    != quality_metrics["distinct_category_names"]
):
    raise ValueError(
        "Silver product category translation contains duplicate category names."
    )

if quality_metrics["missing_english_translations"] > 0:
    raise ValueError(
        "Silver product category translation contains missing English translations."
    )

total_rows,distinct_category_names,missing_category_names,missing_english_translations,distinct_english_translations
71,71,0,0,71


In [0]:
# Step 6. Align and validate schema

product_category_translation_silver_df = (
    product_category_translation_silver_df.select(
        *product_category_translation_silver_schema.fieldNames()
    )
)

if (
    product_category_translation_silver_df.schema
    != product_category_translation_silver_schema
):
    raise ValueError(
        "Silver product category translation schema "
        "does not match the expected schema."
    )

In [0]:
# Step 7. Save the Silver table

(
    product_category_translation_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lakehouse.silver.product_category_name_translation")
)